**Table of contents**<a id='toc00_'></a> 
- [0. Imports](#toc0_)
    - [0.1. Packages and Libraries](#toc0_1_)
    - [0.2. Data](#toc0_2_)
- [1. Base Table Updates](#toc1_)
    - [1.1. Taxonomy](#toc1_1_)
    - [1.2. Realms](#toc1_2_)
    - [1.3. References](#toc1_3_)
- [2. Data Info](#toc2_)
    - [2.1. Observations](#toc2_1_)
    - [2.2. Native Data](#toc2_2_)
- [3. Tables Preparation](#toc3_)
    - [3.1. Native Data](#toc3_1_)
        - [3.1.1. Most Recent Equal Combinations](#toc3_1_1_)
        - [3.1.2. Update Table With Keys](#toc3_1_2_)
            - [3.1.2.1. Realm](#toc3_1_2_1_)
            - [3.1.2.2. Taxonomy](#toc3_1_2_2_)
            - [3.1.2.3. References](#toc3_1_2_3_)


# <a id='toc0_'></a> [0. Imports](#toc00_)

## <a id='toc0_1_'></a> [0.1. Packages and Libraries](#toc0_)

In [166]:
import pandas as pd
import sqlite3

## <a id='toc0_2_'></a> [0.2. Data](#toc0_)

In [167]:
RawData = pd.read_csv(r'Data Raw/ObsList.csv')
RawData.drop_duplicates(inplace=True)
TaxonomyData = pd.read_csv(r'Data Raw/TaxonomyRaw.csv')
TaxonomyData.drop_duplicates(inplace=True)
Regions = pd.read_csv(r'Data Raw/RegionsTableData.csv', sep=';')
Regions.drop_duplicates(inplace=True)
Natives = pd.read_csv(r'Data Raw/NativeRaw.csv', sep=';')
Natives.drop_duplicates(inplace=True)


In [168]:
RawData.head(5)

,Species,Area,Realm,Cryptogenic,Introduced,Dispersal,Established,Eradicated,Intentional_Release,Year,Reference Year,Reference
0,Spodoptera eridania,Brazil,Neotropical,NaN,NaN,NaN,1.0,0.0,NaN,NaN,2024,CAB International (CABI) (2024). CABI Invasive...
1,Phyllocnistis citrella,Brazil,Neotropical,NaN,NaN,NaN,1.0,0.0,NaN,NaN,2024,CAB International (CABI) (2024). CABI Invasive...
2,Hypsipyla grandella,Brazil,Neotropical,NaN,NaN,NaN,1.0,0.0,NaN,NaN,2024,CAB International (CABI) (2024). CABI Invasive...
3,Spodoptera eridania,United States,Nearctic,NaN,NaN,NaN,1.0,0.0,NaN,NaN,2024,CAB International (CABI) (2024). CABI Invasive...
4,Hyphantria cunea,United States,Nearctic,NaN,NaN,NaN,1.0,0.0,NaN,NaN,2024,CAB International (CABI) (2024). CABI Invasive...


In [169]:
TaxonomyData.head(5)

,Species,AcceptedSpecies,Genus,Family
0,Spodoptera eridania,Spodoptera eridania,Spodoptera,Noctuidae
1,Phyllocnistis citrella,Phyllocnistis citrella,Phyllocnistis,Gracillariidae
2,Hypsipyla grandella,Hypsipyla grandella,Hypsipyla,Pyralidae
3,Hyphantria cunea,Hyphantria cunea,Hyphantria,Erebidae
4,Phthorimaea operculella,Phthorimaea operculella,Phthorimaea,Gelechiidae


In [170]:
Regions.head(5)

,AreaID,AreaName,Country,Continent,SubContinent
0,AFG.1_1,Afghanistan,Afghanistan,Asia,NaN
1,AGO.1_1,Angola,Angola,Africa,NaN
2,ALB.1_1,Albania,Albania,Europe,NaN
3,AND.1_1,Andorra,Andorra,Europe,NaN
4,ARE.1_1,United Arab Emirates,United Arab Emirates,Asia,NaN


In [171]:
Natives.head(5)

,Species,AcceptedSpecies,Continent,Realm,Cosmopolitan,Reference Year,Reference
0,Bleszynskia malacelloides,Bleszynskia malacelloides,Australia,Australian,0,2001,"Hoare, R. (2001). Adventive species of Lepidop..."
1,Artigisa melanephele,Artigisa melanephele,Australia,Australian,0,2001,"Hoare, R. (2001). Adventive species of Lepidop..."
2,Acrocercops laciniella,Acrocercops laciniella,Australia,Australian,0,2001,"Hoare, R. (2001). Adventive species of Lepidop..."
3,Zomariana doxasticana,Zomariana doxasticana,Australia,Australian,0,2001,"Hoare, R. (2001). Adventive species of Lepidop..."
4,Coleophora striatipennella,Coleophora striatipennella,Europe,Palearctic,0,2001,"Hoare, R. (2001). Adventive species of Lepidop..."


# <a id='toc1_'></a> [1. Base Tables Updates](#toc00_)

Besides the Areas (Geographical Regions) base table there were no tables fully udpdated on the database, so we started by separating the data that was fully merged on the observation table onto separate parts. Composing: Taxonomy, Realms and References.

## <a id='toc1_1_'></a> [1.1. Taxonomy](#toc1_)

In [172]:
TaxonomyData['SpeciesID'] = 'SP' + (TaxonomyData.index +1).astype(str)

In [173]:
species_to_id = dict(zip(TaxonomyData['Species'], TaxonomyData['SpeciesID']))
TaxonomyData['AcceptedSpeciesID'] = TaxonomyData['AcceptedSpecies'].map(species_to_id)

In [174]:
TaxonomyData.head(5)

,Species,AcceptedSpecies,Genus,Family,SpeciesID,AcceptedSpeciesID
0,Spodoptera eridania,Spodoptera eridania,Spodoptera,Noctuidae,SP1,SP1
1,Phyllocnistis citrella,Phyllocnistis citrella,Phyllocnistis,Gracillariidae,SP2,SP2
2,Hypsipyla grandella,Hypsipyla grandella,Hypsipyla,Pyralidae,SP3,SP3
3,Hyphantria cunea,Hyphantria cunea,Hyphantria,Erebidae,SP4,SP4
4,Phthorimaea operculella,Phthorimaea operculella,Phthorimaea,Gelechiidae,SP5,SP5


## <a id='toc1_2_'></a> [1.2. Realms](#toc1_)

In [175]:
RealmObs = RawData[['Realm']].copy()
RealmNat = Natives[['Realm']].copy()

Realms = pd.concat([RealmObs, RealmNat]).drop_duplicates().reset_index(drop=True)
Realms['RealmID'] = 'RLM' + (Realms.index +1).astype(str)

In [176]:
Realms

,Realm,RealmID
0,Neotropical,RLM1
1,Nearctic,RLM2
2,Oceanina,RLM3
3,Oriental,RLM4
4,Sino-Japanese,RLM5
5,Afrotropical,RLM6
6,Palearctic,RLM7
7,Australian,RLM8
8,Panamanian,RLM9
9,Saharo-Arabian,RLM10


## <a id='toc1_3_'></a> [1.3. References](#toc1_)

In [177]:
ReferencesObs = RawData[['Reference Year', 'Reference']].copy()
ReferencesObs.drop_duplicates(inplace=True)
ReferencesObs.reset_index(drop=True, inplace=True)

In [178]:
ReferencesObs.head(5)

,Reference Year,Reference
0,2024,CAB International (CABI) (2024). CABI Invasive...
1,2020,"Gilligan, T., Brown, J. and Baixeras, J. (2020..."
2,2024,European and Mediterranean Plant Protection Or...
3,2009,"Fodor, E. and Haruta, O. (2009). Niche partiti..."
4,2010,"Heard, T., Elliott, L., Anderson, B., White, L..."


In [179]:
ReferencesNat = Natives[['Reference Year', 'Reference']].copy()
ReferencesNat.drop_duplicates(inplace=True)
ReferencesNat.reset_index(drop=True, inplace=True)

In [180]:
ReferencesNat

,Reference Year,Reference
0,2001,"Hoare, R. (2001). Adventive species of Lepidop..."
1,1992,"Frank, J. and McCoy, E. (1992). Introduction t..."
2,2010,"Lopez-Vaamonde, C., Agassiz, D., Augustin, S.,..."
3,2020,"Gilligan, T., Brown, J. and Baixeras, J. (2020..."
4,2022,"Calhoun, J. and Smith, R. (2022). Brephidium e..."
...,...,...
720,1960,Gozmany L. (1960). The results of the zoologic...
721,2017,"Cock, M. (2017) A preliminary catalogue of the..."
722,1987,"Diakonoff, A. and van Nieukerken, E. (1987). E..."
723,2003,"Baran, T. (2003). Scythris buszkoi sp. n., a n..."


In [181]:
References = pd.concat([ReferencesObs, ReferencesNat]).drop_duplicates().reset_index(drop=True)


In [182]:
References['ReferenceID'] = 'REF' + (References.index +1).astype(str)

In [183]:
References

,Reference Year,Reference,ReferenceID
0,2024,CAB International (CABI) (2024). CABI Invasive...,REF1
1,2020,"Gilligan, T., Brown, J. and Baixeras, J. (2020...",REF2
2,2024,European and Mediterranean Plant Protection Or...,REF3
3,2009,"Fodor, E. and Haruta, O. (2009). Niche partiti...",REF4
4,2010,"Heard, T., Elliott, L., Anderson, B., White, L...",REF5
...,...,...,...
1100,2023,"Assaad, M. (2023). Ecology and impacts of inse...",REF1101
1101,1960,Gozmany L. (1960). The results of the zoologic...,REF1102
1102,2017,"Cock, M. (2017) A preliminary catalogue of the...",REF1103
1103,1987,"Diakonoff, A. and van Nieukerken, E. (1987). E...",REF1104


# <a id='toc2_'></a> [2. Data Info](#toc00_)

## <a id='toc2_1_'></a> [2.1. Observations](#toc2_)

In [184]:
RawData.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16505 entries, 0 to 21198
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Species              16505 non-null  object 
 1   Area                 16505 non-null  object 
 2   Realm                16505 non-null  object 
 3   Cryptogenic          4434 non-null   float64
 4   Introduced           7177 non-null   float64
 5   Dispersal            647 non-null    float64
 6   Established          16305 non-null  float64
 7   Eradicated           16461 non-null  float64
 8   Intentional_Release  4596 non-null   float64
 9   Year                 3057 non-null   float64
 10  Reference Year       16505 non-null  int64  
 11  Reference            16505 non-null  object 
dtypes: float64(7), int64(1), object(4)
memory usage: 1.6+ MB


In [185]:
RawData.describe().T

,count,mean,std,min,25%,50%,75%,max
Cryptogenic,4434.0,0.143888,0.351016,0.0,0.0,0.0,0.0,1.0
Introduced,7177.0,0.970183,0.170095,0.0,1.0,1.0,1.0,1.0
Dispersal,647.0,0.701700,0.457866,0.0,0.0,1.0,1.0,1.0
Established,16305.0,0.968353,0.175063,0.0,1.0,1.0,1.0,1.0
Eradicated,16461.0,0.003888,0.062234,0.0,0.0,0.0,0.0,1.0
Intentional_Release,4596.0,0.054830,0.227673,0.0,0.0,0.0,0.0,1.0
Year,3057.0,1974.614001,51.200735,1565.0,1958.0,1991.0,2008.0,2024.0
Reference Year,16505.0,2020.437201,6.376645,1926.0,2020.0,2024.0,2024.0,2025.0


In [186]:
RawData.describe(include = 'O').T

,count,unique,top,freq
Species,16505,1616,Helicoverpa armigera,362
Area,16505,233,United States,1015
Realm,16505,11,Palearctic,7272
Reference,16505,503,European and Mediterranean Plant Protection Or...,4657


In [187]:
RawData.columns

Index(['Species', 'Area', 'Realm', 'Cryptogenic', 'Introduced', 'Dispersal',
       'Established', 'Eradicated', 'Intentional_Release', 'Year',
       'Reference Year', 'Reference'],
      dtype='object')

## <a id='toc2_2_'></a> [2.2. Native Data](#toc2_)

In [188]:
Natives.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2546 entries, 0 to 7749
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Species          2546 non-null   object
 1   AcceptedSpecies  2546 non-null   object
 2   Continent        2544 non-null   object
 3   Realm            2546 non-null   object
 4   Cosmopolitan     2546 non-null   int64 
 5   Reference Year   2546 non-null   int64 
 6   Reference        2546 non-null   object
dtypes: int64(2), object(5)
memory usage: 159.1+ KB


In [189]:
Natives.describe().T

,count,mean,std,min,25%,50%,75%,max
Cosmopolitan,2546.0,0.001571,0.039614,0.0,0.0,0.0,0.0,1.0
Reference Year,2546.0,2006.905342,20.125541,1775.0,2003.0,2011.0,2020.0,2025.0


In [190]:
Natives.describe(include = 'O').T

,count,unique,top,freq
Species,2546,920,Lampides boeticus,14
AcceptedSpecies,2546,909,Lampides boeticus,14
Continent,2544,7,Asia,888
Realm,2546,11,Palearctic,807
Reference,2546,725,"Khramov, P. (Ed.) (2007). Insecta.pro: interna...",154


In [191]:
Natives.columns

Index(['Species', 'AcceptedSpecies', 'Continent', 'Realm', 'Cosmopolitan',
       'Reference Year', 'Reference'],
      dtype='object')

# <a id='toc3_'></a> [3. Tables Preparation](#toc00_)

Before we start inputing data onto the database we need to prepare the tables for all of them to have the predefined structure. We needed to update the fields for it to have the connections with the ID's. 

## <a id='toc3_1_'></a> [3.1. Native Data](#toc3_)

In [192]:
Natives_DB = Natives[['Species', 'AcceptedSpecies', 'Continent', 'Realm', 'Cosmopolitan', 'Reference Year', 'Reference']].copy()

### <a id='toc3_1_1_'></a> [3.1.1. Most Recent Equal Combinations](#toc3_1_)

In [193]:
idx = Natives_DB.groupby(['AcceptedSpecies', 'Continent', 'Realm', 'Cosmopolitan'])['Reference Year'].idxmax()

Natives_DB = Natives_DB.loc[idx].reset_index(drop=True)


### <a id='toc3_1_2_'></a> [3.1.2. Update Table with Keys](#toc3_1_)

#### <a id='toc3_1_2_1_'></a> [3.1.2.1. Realm](#toc3_1_2_)

In [194]:
Natives_DB = Natives_DB.merge(Realms, on='Realm', how='left')
Natives_DB.drop(columns='Realm', inplace=True)

In [195]:
Natives_DB

,Species,AcceptedSpecies,Continent,Cosmopolitan,Reference Year,Reference,RealmID
0,Abaeis nicippe,Abaeis nicippe,North America,0,2024,"Lotts, K. and Naberhaus, T. (coords.) (2024). ...",RLM2
1,Abaeis nicippe,Abaeis nicippe,North America,0,2024,"Lotts, K. and Naberhaus, T. (coords.) (2024). ...",RLM9
2,Acalyptris platani,Acalyptris platani,Africa,0,2003,"Sefrova, H. (2003). Invasions of Lithocolletin...",RLM10
3,Acalyptris platani,Acalyptris platani,Asia,0,2007,"van Nieukerken, E. (2007). Acalyptris Meyrick:...",RLM7
4,Acalyptris platani,Acalyptris platani,Asia,0,2007,"van Nieukerken, E. (2007). Acalyptris Meyrick:...",RLM10
...,...,...,...,...,...,...,...
2195,Zizula hylax,Zizula hylax,Australia,0,2013,"Fric, Z. and Hula, V. (2013). Zizula hylax (F...",RLM8
2196,Zizula hylax,Zizula hylax,Oceania,0,2013,"Fric, Z. and Hula, V. (2013). Zizula hylax (F...",RLM3
2197,Zomariana doxasticana,Zomariana doxasticana,Australia,0,2001,"Hoare, R. (2001). Adventive species of Lepidop...",RLM8
2198,Zophodia lucidalis,Zophodia lucidalis,North America,0,1990,"Habeck, D. and Bennett, F. (1990). Cactoblasti...",RLM9


#### <a id='toc3_1_2_2_'></a> [3.1.2.2. Taxonomy](#toc3_1_2_)

In [196]:
Natives_DB.drop(columns='AcceptedSpecies', inplace=True)
Natives_DB = Natives_DB.merge(TaxonomyData, on='Species', how='left')

In [197]:
Natives_DB.drop(columns=['Species', 'AcceptedSpecies', 'Genus', 'Family', 'AcceptedSpeciesID'], inplace=True)

In [198]:
Natives_DB

,Continent,Cosmopolitan,Reference Year,Reference,RealmID,SpeciesID
0,North America,0,2024,"Lotts, K. and Naberhaus, T. (coords.) (2024). ...",RLM2,SP405
1,North America,0,2024,"Lotts, K. and Naberhaus, T. (coords.) (2024). ...",RLM9,SP405
2,Africa,0,2003,"Sefrova, H. (2003). Invasions of Lithocolletin...",RLM10,SP1015
3,Asia,0,2007,"van Nieukerken, E. (2007). Acalyptris Meyrick:...",RLM7,SP1015
4,Asia,0,2007,"van Nieukerken, E. (2007). Acalyptris Meyrick:...",RLM10,SP1015
...,...,...,...,...,...,...
2195,Australia,0,2013,"Fric, Z. and Hula, V. (2013). Zizula hylax (F...",RLM8,SP690
2196,Oceania,0,2013,"Fric, Z. and Hula, V. (2013). Zizula hylax (F...",RLM3,SP690
2197,Australia,0,2001,"Hoare, R. (2001). Adventive species of Lepidop...",RLM8,SP927
2198,North America,0,1990,"Habeck, D. and Bennett, F. (1990). Cactoblasti...",RLM9,NaN


#### <a id='toc3_1_2_3_'></a> [3.1.2.3. References](#toc3_1_2_)

In [199]:
Natives_DB.drop(columns='Reference Year', inplace=True)
Natives_DB = Natives_DB.merge(References, on='Reference', how='left')

In [200]:
Natives_DB.drop(columns=['Reference', 'Reference Year'], inplace=True)

In [201]:
Natives_DB

,Continent,Cosmopolitan,RealmID,SpeciesID,ReferenceID
0,North America,0,RLM2,SP405,REF505
1,North America,0,RLM9,SP405,REF505
2,Africa,0,RLM10,SP1015,REF148
3,Asia,0,RLM7,SP1015,REF506
4,Asia,0,RLM10,SP1015,REF506
...,...,...,...,...,...
2195,Australia,0,RLM8,SP690,REF972
2196,Oceania,0,RLM3,SP690,REF972
2197,Australia,0,RLM8,SP927,REF101
2198,North America,0,RLM9,NaN,REF934
